# Inkapsling (Encapsulation)
### Programmering 2 — exempel

Den här filen innehåller **10 exempel** om inkapsling. Alla exempel bygger på en klass `Bankkonto`, som steg för steg blir bättre på att skydda sin egen data.

**Vad är inkapsling?**
Inkapsling betyder att ett objekt **skyddar sin egen data** och själv bestämmer hur den får läsas och ändras. Istället för att resten av programmet ändrar attributen direkt går allt via metoder som kan kontrollera att värdena är rimliga.

En bra liknelse är en bankomat: du kan sätta in och ta ut pengar via knapparna (metoderna), men du kan inte öppna maskinen och ändra siffrorna i bankens system direkt.

**Så här jobbar du:**
1. Läs textrutan före varje exempel.
2. Kör kodcellen (Shift + Enter) och titta på resultatet.
3. Läs förklaringen efter koden, och prova gärna förslaget under *Testa själv*.

Varje kodcell är fristående, så du kan köra dem i valfri ordning.

---
### Exempel 1 — Problemet: vem som helst kan ändra datan

Hittills har vi skrivit klasser där alla attribut går att ändra direkt utifrån, till exempel `konto.saldo = 500`. Det ser smidigt ut, men det betyder att **vem som helst kan sätta vilket värde som helst**, även värden som inte är rimliga.

Tänk dig en riktig bank. Skulle det vara okej om ett program kunde skriva `konto.saldo = -5000` utan att någon kontrollerade det? Knappast.

**Inkapsling** handlar om att ett objekt ska *skydda sin egen data*. Andra delar av programmet ska inte peta direkt i attributen, utan gå via metoder som klassen själv har skrivit. Då kan klassen kontrollera att allt blir rätt.

Kör koden nedan och se hur lätt det är att förstöra ett konto när det saknas inkapsling.

In [ ]:
class Bankkonto:
    def __init__(self, agare, saldo):
        self.agare = agare      # vem som äger kontot
        self.saldo = saldo      # hur mycket pengar som finns

konto = Bankkonto("Lisa", 1000)
print(konto.agare, "har", konto.saldo, "kr")

# Här ändrar vi saldot direkt, utan att någon kontrollerar värdet.
# Python protesterar inte alls, fast ett saldo på -5000 kr är orimligt.
konto.saldo = -5000
print(konto.agare, "har", konto.saldo, "kr")

# Det går till och med att sätta saldot till en text, vilket förstör allt
# om vi senare försöker räkna med det.
konto.saldo = "massor"
print(konto.agare, "har", konto.saldo, "kr")

**Vad händer här?**

- Klassen `Bankkonto` har två vanliga attribut, `agare` och `saldo`.
- Efter att objektet skapats ändrar vi `konto.saldo` direkt, först till `-5000` och sedan till texten `"massor"`.
- Python gör precis som vi säger. Klassen har ingen möjlighet att säga nej, eftersom ingen av dess metoder är inblandad.

**Poängen:** Så länge attributen är öppna för alla kan klassen inte garantera att datan är korrekt. I resten av exemplen ska vi steg för steg bygga ett konto som skyddar sig självt.

---
### Exempel 2 — Ett understreck: "rör inte det här"

I många språk, som Java och C#, finns nyckelordet `private` som gör ett attribut helt osynligt utifrån. Python fungerar annorlunda: där bygger man istället på en **överenskommelse mellan programmerare**.

Om ett attributnamn börjar med **ett understreck**, till exempel `_saldo`, betyder det:

> *"Det här är klassens interna data. Använd det inte utanför klassen."*

Python stoppar dig inte tekniskt, men alla som läser koden förstår att attributet är "privat". Attribut **utan** understreck, som `agare`, är **publika** och får användas fritt.

Ett attribut med ett understreck kallas ibland **skyddat** (protected).

In [ ]:
class Bankkonto:
    def __init__(self, agare, saldo):
        self.agare = agare      # publikt attribut: får läsas och ändras utifrån
        self._saldo = saldo     # skyddat attribut: bara klassen själv ska röra det

    def visa_saldo(self):
        # Inuti klassen är det helt i sin ordning att använda self._saldo.
        print(f"{self.agare} har {self._saldo} kr")

konto = Bankkonto("Lisa", 1000)
konto.visa_saldo()

# Vi använder bara publika saker utanför klassen:
print("Kontoägare:", konto.agare)

# Raden nedan skulle fungera tekniskt, men den bryter mot överenskommelsen.
# En programmerare som ser  konto._saldo  i koden vet direkt att något är fel.
# konto._saldo = -5000

**Vad händer här?**

- `self._saldo` har ett understreck först. Det är en signal till alla som läser koden.
- Metoden `visa_saldo()` ligger **inuti** klassen och får därför använda `_saldo`.
- Utanför klassen använder vi bara `agare` (publikt) och metoden `visa_saldo()`.

**Kom ihåg:** Understrecket är ett *löfte*, inte ett lås. Men det är så man arbetar i Python, och du kommer att se det överallt i riktig kod.

**Testa själv:** Ta bort `#` framför sista raden och kör igen. Det fungerar, men nu har du brutit mot regeln.

---
### Exempel 3 — Metoder som enda vägen in

Om `_saldo` inte ska ändras utifrån, hur sätter man då in eller tar ut pengar? Svaret är att klassen erbjuder **metoder** för det. Metoderna blir objektets *gränssnitt*: de enda dörrarna in till datan.

Det fina med metoder är att de kan **kontrollera (validera)** värdet innan det sparas. En insättning på `-200` kr ska till exempel inte godkännas, och man ska inte kunna ta ut mer pengar än det finns på kontot.

Titta särskilt på `if`, `elif` och `else` i metoderna nedan. Det är där skyddet sitter.

In [ ]:
class Bankkonto:
    def __init__(self, agare, saldo=0):
        self.agare = agare
        self._saldo = saldo

    def satt_in(self, belopp):
        # Kontroll: man kan bara sätta in positiva belopp.
        if belopp <= 0:
            print("Fel: beloppet måste vara större än 0.")
        else:
            self._saldo = self._saldo + belopp
            print(f"Satte in {belopp} kr.")

    def ta_ut(self, belopp):
        # Två kontroller: beloppet måste vara positivt
        # och det måste finnas tillräckligt med pengar.
        if belopp <= 0:
            print("Fel: beloppet måste vara större än 0.")
        elif belopp > self._saldo:
            print(f"Fel: du försöker ta ut {belopp} kr men har bara {self._saldo} kr.")
        else:
            self._saldo = self._saldo - belopp
            print(f"Tog ut {belopp} kr.")

    def visa_saldo(self):
        print(f"{self.agare} har {self._saldo} kr")


konto = Bankkonto("Lisa", 1000)

konto.satt_in(500)      # godkänns
konto.ta_ut(3000)       # nekas: för lite pengar
konto.satt_in(-200)     # nekas: negativt belopp
konto.ta_ut(200)        # godkänns

konto.visa_saldo()      # 1000 + 500 - 200 = 1300

**Vad händer här?**

1. `satt_in(500)` går igenom kontrollen och saldot blir 1500.
2. `ta_ut(3000)` stoppas av `elif`, eftersom 3000 är mer än saldot.
3. `satt_in(-200)` stoppas av `if belopp <= 0`.
4. `ta_ut(200)` godkänns och saldot blir 1300.

**Poängen:** Nu är det *klassen* som bestämmer vilka ändringar som är tillåtna. Det spelar ingen roll hur resten av programmet ser ut, för saldot kan aldrig bli negativt så länge alla använder metoderna.

**Testa själv:** Försök få saldot negativt med bara `satt_in()` och `ta_ut()`. Det ska inte gå.

---
### Exempel 4 — Getter: en metod som lämnar ut värdet

Ibland behöver resten av programmet **läsa** ett skyddat värde, till exempel för att räkna med saldot. Då skriver man en metod som *returnerar* värdet. En sådan metod kallas en **getter** (av engelskans *get*, hämta).

Skillnaden mot `visa_saldo()` från förra exemplet:

| `visa_saldo()` | `hamta_saldo()` |
|---|---|
| **skriver ut** en text med `print` | **returnerar** talet med `return` |
| bra för att visa för användaren | bra när koden ska räkna eller jämföra |

Kom ihåg skillnaden mellan `print` och `return` från OOP-grunderna!

In [1]:
class Bankkonto:
    def __init__(self, agare, saldo=0):
        self.agare = agare
        self._saldo = saldo

    def hamta_saldo(self):
        # En getter: lämnar ut värdet men låter det vara oförändrat.
        return self._saldo

    def satt_in(self, belopp):
        if belopp > 0:
            self._saldo = self._saldo + belopp


konto = Bankkonto("Lisa", 1000)
konto.satt_in(250)

# Vi hämtar värdet och kan sedan räkna med det som med vilket tal som helst.
saldo = konto.hamta_saldo()
print("Saldo:", saldo, "kr")
print("Dubbla saldot vore:", saldo * 2, "kr")

# Vi kan också använda gettern direkt i ett villkor.
if konto.hamta_saldo() >= 1000:
    print("Du har råd med en ny cykel!")

Saldo: 1250 kr
Dubbla saldot vore: 2500 kr
Du har råd med en ny cykel!


**Vad händer här?**

- `hamta_saldo()` returnerar `self._saldo`. Den *läser* bara, den ändrar ingenting.
- Eftersom värdet returneras kan vi spara det i en variabel, räkna med det och använda det i ett `if`.
- Den som använder klassen kan alltså *se* saldot, men bara *ändra* det via `satt_in()`.

**Poängen:** Med en getter bestämmer klassen själv hur datan lämnas ut. Man kan till exempel välja att bara ha en getter och ingen metod för att ändra värdet. Då blir det **skrivskyddat** utifrån.

---
### Exempel 5 — Setter: en metod som kontrollerar nya värden

Motsatsen till en getter är en **setter** (av engelskans *set*, sätta): en metod som tar emot ett nytt värde, kontrollerar det och först därefter sparar det.

Här gör vi ägarnamnet skyddat (`_agare`) och lägger till en setter som vägrar ta emot ett tomt namn. Ett konto utan ägare vore ju meningslöst.

Lägg märke till mönstret, som du kommer att använda om och om igen:

1. Ta emot det nya värdet som parameter.
2. Kontrollera om det är giltigt.
3. Om det är giltigt: spara det. Annars: skriv ett felmeddelande och låt det gamla värdet vara kvar.

In [ ]:
class Bankkonto:
    def __init__(self, agare, saldo=0):
        self._agare = agare
        self._saldo = saldo

    def hamta_agare(self):          # getter
        return self._agare

    def satt_agare(self, nytt_namn):  # setter
        if nytt_namn == "":
            print("Fel: ägaren måste ha ett namn.")
        else:
            self._agare = nytt_namn


konto = Bankkonto("Lisa", 1000)
print("Ägare:", konto.hamta_agare())

konto.satt_agare("Lisa Karlsson")   # giltigt, sparas
print("Ägare:", konto.hamta_agare())

konto.satt_agare("")                # ogiltigt, nekas
print("Ägare:", konto.hamta_agare())  # det gamla namnet finns kvar

**Vad händer här?**

- `satt_agare("Lisa Karlsson")` passerar kontrollen och namnet uppdateras.
- `satt_agare("")` stoppas. Felmeddelandet skrivs ut och `_agare` behåller sitt gamla värde.

**Tänk på namngivningen:** En setter *sätter* värdet till det nya värdet: `self._agare = nytt_namn`. Den ska inte lägga ihop det gamla och det nya. Det här är samma sak som i övningen med `Termometer` och `satt_temperatur()`.

**Testa själv:** Lägg till en kontroll som också nekar namn som är kortare än 2 tecken. Tips: `len(nytt_namn)`.

---
### Exempel 6 — `@property`: en getter som ser ut som ett attribut

Getters och setters fungerar bra, men det blir lite klumpigt att skriva `konto.hamta_saldo()` överallt. Python har därför ett smidigare sätt: **`@property`**.

Om man skriver `@property` ovanför en metod kan den användas **som ett attribut**, alltså utan parenteser:

```python
print(konto.saldo)      # istället för konto.hamta_saldo()
```

Bakom kulisserna är det fortfarande en metod som körs, så klassen har kvar kontrollen. Och eftersom vi *bara* har skrivit en getter går det inte att ändra värdet. Attributet blir **skrivskyddat**.

Tecknet `@` framför `property` kallas en **dekorator**. Den "dekorerar" metoden med extra beteende.

In [ ]:
class Bankkonto:
    def __init__(self, agare, saldo=0):
        self.agare = agare
        self._saldo = saldo          # den riktiga datan, skyddad

    @property
    def saldo(self):                 # samma namn som vi vill använda utifrån
        return self._saldo

    def satt_in(self, belopp):
        if belopp > 0:
            self._saldo = self._saldo + belopp


konto = Bankkonto("Lisa", 1000)
konto.satt_in(500)

# Inga parenteser! Det ser ut som ett vanligt attribut.
print("Saldo:", konto.saldo, "kr")

# Men det går INTE att ändra saldot direkt längre.
# Vi använder try/except så att programmet inte kraschar när vi testar.
try:
    konto.saldo = -5000
except AttributeError:
    print("Nekat! saldo är skrivskyddat och kan bara ändras via metoderna.")

print("Saldo fortfarande:", konto.saldo, "kr")

**Vad händer här?**

- `_saldo` (med understreck) är den riktiga datan inuti objektet.
- `saldo` (utan understreck) är en *property* som returnerar `_saldo`.
- `konto.saldo` fungerar alltså som ett attribut när man **läser**.
- `konto.saldo = -5000` ger ett `AttributeError`, eftersom det saknas en setter. Vi fångar felet med `try`/`except`, som ni har sett i Programmering 1.

**Viktigt:** Propertyn och det skyddade attributet får **inte** heta exakt samma sak. Därför heter den ena `saldo` och den andra `_saldo`.

**Testa själv:** Ta bort `try`, `except` och indraget runt `konto.saldo = -5000` och kör. Läs felmeddelandet.

---
### Exempel 7 — `@property` med setter

Vill man att ett värde ska gå att **ändra** med vanlig attribut-syntax, men ändå kontrolleras, lägger man till en **setter** till sin property.

Den skrivs så här:

```python
@agare.setter
def agare(self, nytt_namn):
    ...
```

Namnet före `.setter` måste vara samma som propertyns namn. Nu gör vi samma sak som i exempel 5, men med snyggare syntax. Jämför gärna exemplen!

In [ ]:
class Bankkonto:
    def __init__(self, agare, saldo=0):
        self._agare = agare
        self._saldo = saldo

    @property
    def agare(self):                 # getter
        return self._agare

    @agare.setter
    def agare(self, nytt_namn):      # setter, körs vid  konto.agare = ...
        if nytt_namn == "":
            print("Fel: ägaren måste ha ett namn.")
        else:
            self._agare = nytt_namn


konto = Bankkonto("Lisa", 1000)
print("Ägare:", konto.agare)

konto.agare = "Lisa Karlsson"    # ser ut som en vanlig tilldelning...
print("Ägare:", konto.agare)

konto.agare = ""                 # ...men settern stoppar ogiltiga värden
print("Ägare:", konto.agare)

**Vad händer här?**

- `konto.agare` (läsa) kör getter-metoden.
- `konto.agare = "Lisa Karlsson"` (skriva) kör **setter**-metoden. Python skickar automatiskt in värdet till höger om `=` som parametern `nytt_namn`.
- Settern kontrollerar värdet precis som `satt_agare()` i exempel 5.

**Jämför:**

| Exempel 5 | Exempel 7 |
|---|---|
| `konto.hamta_agare()` | `konto.agare` |
| `konto.satt_agare("Lisa")` | `konto.agare = "Lisa"` |

Båda sätten är inkapsling. `@property` är det sätt som oftast används i Python, eftersom koden blir lättare att läsa.

---
### Exempel 8 — Beräknade properties

En property behöver inte bara lämna ut ett sparat attribut. Den kan också **räkna fram** ett värde utifrån objektets data varje gång man frågar.

Här har kontot en property `ar_tomt` som svarar `True` eller `False`, och en property `saldo_euro` som räknar om saldot till euro. Inget av dem sparas som ett eget attribut. De räknas fram när de behövs och kan därför aldrig bli fel eller gamla.

Sådana properties har ingen setter, eftersom det inte är meningsfullt att *sätta* något som räknas fram.

In [ ]:
class Bankkonto:
    EURO_KURS = 11.50                # klassattribut: kronor per euro

    def __init__(self, agare, saldo=0):
        self.agare = agare
        self._saldo = saldo

    @property
    def saldo(self):
        return self._saldo

    @property
    def ar_tomt(self):
        # Räknas fram varje gång: True om saldot är 0, annars False.
        return self._saldo == 0

    @property
    def saldo_euro(self):
        # Räknas också fram, avrundat till två decimaler.
        return round(self._saldo / Bankkonto.EURO_KURS, 2)

    def ta_ut(self, belopp):
        if 0 < belopp <= self._saldo:
            self._saldo = self._saldo - belopp


konto = Bankkonto("Lisa", 1150)
print("Saldo:", konto.saldo, "kr")
print("I euro:", konto.saldo_euro)
print("Är kontot tomt?", konto.ar_tomt)

konto.ta_ut(1150)
print("Efter uttag. Är kontot tomt?", konto.ar_tomt)

**Vad händer här?**

- `ar_tomt` jämför `_saldo` med 0 och returnerar resultatet (`True` eller `False`).
- `saldo_euro` delar saldot med växelkursen.
- När saldot ändras ändras svaren automatiskt, eftersom de räknas fram på nytt varje gång.

**Poängen:** Den som använder klassen behöver inte veta *hur* värdet räknas fram. Det är också en del av inkapslingen: detaljerna göms inuti klassen, och utifrån syns bara ett enkelt namn.

**Testa själv:** Lägg till en property `ar_rik` som returnerar `True` om saldot är minst 100 000 kr.

---
### Exempel 9 — Två understreck: namnförvanskning

Om ett attributnamn börjar med **två understreck**, till exempel `__pinkod`, gör Python något extra: det byter automatiskt namn på attributet till `_Bankkonto__pinkod`. Det kallas **namnförvanskning** (på engelska *name mangling*).

Effekten är att `konto.__pinkod` inte längre hittas utifrån, så det blir svårare att komma åt värdet av misstag. Det är fortfarande inte ett riktigt lås (den som verkligen vill kan skriva det förvanskade namnet), men det är ett starkare skydd än ett understreck.

**Tumregel:**

| Namn | Betydelse |
|---|---|
| `saldo` | publikt: får användas fritt |
| `_saldo` | skyddat: använd bara inuti klassen (och i subklasser) |
| `__pinkod` | privat: bara klassen själv, Python byter namn på det |

**Obs!** Namn som både börjar *och* slutar med två understreck, som `__init__` och `__str__`, är något helt annat: det är Pythons egna specialmetoder.

In [ ]:
class Bankkonto:
    def __init__(self, agare, saldo, pinkod):
        self.agare = agare
        self._saldo = saldo
        self.__pinkod = pinkod           # två understreck: extra skyddat

    def kontrollera_pinkod(self, gissning):
        # Bara klassen själv jämför med den riktiga koden.
        # Pinkoden lämnas aldrig ut, vi svarar bara True eller False.
        return gissning == self.__pinkod


konto = Bankkonto("Lisa", 1000, 1234)

print("Rätt kod 1111?", konto.kontrollera_pinkod(1111))
print("Rätt kod 1234?", konto.kontrollera_pinkod(1234))

# Försöker vi läsa __pinkod utifrån hittar Python inget sådant attribut.
try:
    print(konto.__pinkod)
except AttributeError:
    print("Hittade inget attribut som heter __pinkod!")

# Python har bytt namn på det. Det här fungerar, men gör aldrig så i riktig kod:
print("Det förvanskade namnet:", konto._Bankkonto__pinkod)

**Vad händer här?**

- `self.__pinkod` sparas inuti objektet under namnet `_Bankkonto__pinkod`.
- Metoden `kontrollera_pinkod()` ligger inuti klassen och kommer åt det utan problem.
- Utifrån ger `konto.__pinkod` ett `AttributeError`.
- Sista raden visar att skyddet inte är absolut. Det finns för att förhindra misstag, inte för att stoppa den som medvetet vill fuska.

**När ska man använda två understreck?** Sällan. För det mesta räcker ett understreck. Två understreck används främst när man vill vara säker på att en subklass inte råkar skriva över attributet (se nästa exempel).

---
### Exempel 10 — Inkapsling och arv

Nu kombinerar vi inkapsling med arv. `Sparkonto` ärver från `Bankkonto` och lägger till ränta.

Frågan är: **får subklassen använda förälderns skyddade attribut?**

- Attribut med **ett** understreck (`_saldo`) får användas i subklasser. Det räknas som "inne i familjen".
- Attribut med **två** understreck (`__pinkod`) går inte att nå direkt från subklassen, eftersom namnet har förvanskats till `_Bankkonto__pinkod`.

Här använder `Sparkonto` `_saldo` i sin egen metod `lagg_till_ranta()`, och samtidigt ärver den propertyn `saldo` och metoden `satt_in()` från `Bankkonto`.

In [ ]:
class Bankkonto:
    def __init__(self, agare, saldo=0):
        self.agare = agare
        self._saldo = saldo

    @property
    def saldo(self):
        return self._saldo

    def satt_in(self, belopp):
        if belopp > 0:
            self._saldo = self._saldo + belopp
        else:
            print("Fel: beloppet måste vara större än 0.")


class Sparkonto(Bankkonto):
    def __init__(self, agare, saldo=0, ranta=0.03):
        super().__init__(agare, saldo)   # låt Bankkonto sätta agare och _saldo
        self._ranta = ranta              # nytt skyddat attribut, bara i Sparkonto

    @property
    def ranta(self):
        return self._ranta

    def lagg_till_ranta(self):
        # Subklassen får använda _saldo (ett understreck) från föräldern.
        ranta_kr = self._saldo * self._ranta
        self._saldo = self._saldo + ranta_kr
        print(f"Räntan blev {ranta_kr} kr.")


spar = Sparkonto("Lisa", 10000, 0.02)
spar.satt_in(5000)                   # ärvd metod från Bankkonto
spar.lagg_till_ranta()               # egen metod i Sparkonto
print("Saldo:", spar.saldo, "kr")    # ärvd property från Bankkonto
print("Räntesats:", spar.ranta)

**Vad händer här?**

1. `super().__init__(agare, saldo)` låter `Bankkonto` sätta upp `agare` och `_saldo`, precis som i arvsavsnittet.
2. `Sparkonto` lägger till det egna skyddade attributet `_ranta` och en property för att läsa det.
3. `lagg_till_ranta()` använder `self._saldo` direkt. Det är tillåtet, eftersom `Sparkonto` är en subklass.
4. Utanför klasserna används bara `saldo`, `ranta`, `satt_in()` och `lagg_till_ranta()`, alltså det publika gränssnittet.

**Sammanfattning av hela avsnittet:**

- Skydda datan med understreck (`_attribut`).
- Låt metoder eller properties vara vägen in och ut.
- Kontrollera värden innan de sparas (validering).
- Använd `@property` för en snygg syntax med bibehållen kontroll.

Då kan objektet aldrig hamna i ett ogiltigt läge, oavsett vem som använder klassen.

---
## Sammanfattning

| Begrepp | Hur det ser ut | Vad det betyder |
|---|---|---|
| Publikt attribut | `self.agare` | Får användas fritt utifrån |
| Skyddat attribut | `self._saldo` | Använd bara i klassen och subklasser |
| Privat attribut | `self.__pinkod` | Python byter namn på det, extra skyddat |
| Getter | `def hamta_saldo(self): return self._saldo` | Lämnar ut värdet |
| Setter | `def satt_agare(self, nytt): ...` | Kontrollerar och sparar ett nytt värde |
| Property | `@property` | Getter som används utan parenteser |
| Property-setter | `@agare.setter` | Setter som används med `=` |

**Den viktigaste regeln:** Ett objekt ska aldrig kunna hamna i ett ogiltigt läge. Kontrollera därför värden innan de sparas.